# Identifying public CF airway metagenomes

In [ ]:
import duckdb

%load_ext sql
conn = duckdb.connect()
%sql conn --alias duckdb

In [ ]:
%%sql
INSTALL httpfs;
LOAD httpfs;
INSTALL parquet;
LOAD parquet;
COPY (
  SELECT 
    acc, biosample, consent,
    bioproject, librarysource, libraryselection, librarylayout,
    platform, instrument, organism, mbases, avgspotlen, assay_type
  FROM read_parquet('s3://sra-pub-metadata-us-east-1/sra/metadata/*')
  WHERE assay_type != 'AMPLICON'
  AND consent = 'public'
  AND platform = 'ILLUMINA'
  AND mbases >= 10
  AND libraryselection != 'PCR'
  AND (librarysource = 'METAGENOMIC'
    OR librarysource = 'METATRANSCRIPTOMIC'
    OR organism LIKE '%microbiom%'
    OR organism LIKE '%metagenom%'
    OR organism LIKE '%metatran%')
  AND bioproject IN (
    'PRJEB52664',
    'PRJEB51981',
    'PRJEB85425',
    'PRJNA1182815',
    'PRJNA1176118',
    'PRJNA1152493',
    'PRJEB56242',
    'PRJNA1126024',
    'PRJNA1119982',
    'PRJEB75534',
    'PRJNA1112360',
    'PRJNA1101448',
    'PRJNA1091195',
    'PRJEB51530',
    'PRJNA1081394',
    'PRJEB67893',
    'PRJEB64339',
    'PRJNA1055940',
    'PRJEB69226',
    'PRJNA1033679',
    'PRJNA987026',
    'PRJNA978345',
    'PRJNA935808',
    'PRJNA934375',
    'PRJNA931830',
    'PRJEB54014',
    'PRJEB52683',
    'PRJNA870948',
    'PRJNA861321',
    'PRJEB52317',
    'PRJEB52482',
    'PRJEB52599',
    'PRJNA839435',
    'PRJNA825831',
    'PRJNA824119',
    'PRJNA810321',
    'PRJEB50802',
    'PRJEB49871',
    'PRJNA788640',
    'PRJNA769290',
    'PRJNA759072',
    'PRJNA733203',
    'PRJNA717158',
    'PRJNA714872',
    'PRJEB41644',
    'PRJNA692838',
    'PRJNA667146',
    'PRJNA666737',
    'PRJNA666192',
    'PRJNA662963',
    'PRJEB38221',
    'PRJNA647768',
    'PRJNA645089',
    'PRJNA644285',
    'PRJNA644204',
    'PRJNA641924',
    'PRJEB31332',
    'PRJNA629006',
    'PRJEB32062',
    'PRJNA615628',
    'PRJNA613708',
    'PRJNA611611',
    'PRJNA599290',
    'PRJNA591432',
    'PRJNA590873',
    'PRJEB33434',
    'PRJEB29375',
    'PRJEB33064',
    'PRJNA548026',
    'PRJNA545010',
    'PRJNA544933',
    'PRJNA544655',
    'PRJNA530252',
    'PRJNA520921',
    'PRJNA516870',
    'PRJNA516442',
    'PRJNA514329',
    'PRJEB28588',
    'PRJEB28553',
    'PRJNA503799',
    'PRJEB28158',
    'PRJEB28311',
    'PRJEB20836',
    'PRJEB28175',
    'PRJNA480715',
    'PRJEB27575',
    'PRJEB27573',
    'PRJNA450612',
    'PRJNA450593',
    'PRJNA437613',
    'PRJNA433469',
    'PRJEB49208',
    'PRJNA374847',
    'PRJNA360332',
    'PRJNA339813',
    'PRJNA339795',
    'PRJEB10599',
    'PRJEB14440',
    'PRJEB14149',
    'PRJEB14304',
    'PRJEB13657',
    'PRJNA318795',
    'PRJNA316588',
    'PRJNA316056',
    'PRJNA314903',
    'PRJNA311065',
    'PRJEB11908',
    'PRJNA305156',
    'PRJNA290694',
    'PRJNA288589',
    'PRJNA288476',
    'PRJEB7867',
    'PRJNA271691',
    'PRJNA269267',
    'PRJNA258440',
    'PRJNA258369',
    'PRJNA258130',
    'PRJNA258127',
    'PRJNA257190',
    'PRJNA256169',
    'PRJEB406',
    'PRJNA254115',
    'PRJNA252825',
    'PRJNA246028',
    'PRJNA244556',
    'PRJNA239463',
    'PRJNA238324',
    'PRJNA235284',
    'PRJNA234009',
    'PRJNA232392',
    'PRJEB4820',
    'PRJNA207555',
    'PRJNA193431',
    'PRJNA178068',
    'PRJNA172839',
    'PRJNA167673',
    'PRJNA71831',
    'PRJNA66313',
    'PRJNA63141',
    'PRJNA39545',
    'PRJNA28593',
    'PRJNA28441',
    'PRJNA846291',
    'PRJNA510441',
    'PRJEB51171',
    'PRJNA580503',
    'PRJNA741386',
    'PRJNA437845',
    'PRJNA510441',
    'PRJNA573047',
    'PRJEB24688',
    'PRJNA587705',
    'PRJNA909326',
    'PRJNA741386')
) TO '2025_08_26_sra_metadata.parquet' (FORMAT 'parquet');

In [ ]:
import polars as pl
pl.Config(tbl_rows=25)

# read sra parquet file
sra_meta = (
    pl.read_parquet('2025_08_26_sra_metadata.parquet',
        columns=['acc', 'biosample', 'bioproject', 'organism', 'librarylayout', 'platform', 'mbases', 'avgspotlen']
    )
    .with_columns([
        pl.col('mbases').cast(pl.Float64),
        pl.col('avgspotlen').cast(pl.Float64),
    ])
)

# list different organisms to identify false hits
print(sra_meta[['organism']].unique())

In [ ]:
# remove non-oral metagenome organisms
sra_org_filt = (
    sra_meta
        .filter(~pl.col('organism').is_in([
            'Pseudomonas aeruginosa',
            'Pseudomonas aeruginosa PAO1',
            'mouse metagenome',
            'Staphylococcus phage SWC012',
            'Mycobacteroides abscessus',
            'Pseudoalteromonas atlantica',
            'human gut metagenome',
            'synthetic metagenome',
            'mixed culture metagenome',
            'human feces metagenome'])
        )
)
print("Number of organism filtered accessions: ", sra_org_filt.shape[0])

In [ ]:
# investigate ambiguous organisms for further filtering
print(
    sra_org_filt
        .filter(pl.col('organism').is_in(['human metagenome', 'Homo sapiens', 'metagenome']))
        ['bioproject'].unique()
)

# PRJNA71831; these are all from CF lungs       KEEP
# PRJEB28158; has CF, ID, and COPD samples,     REMOVE
# PRJEB50802; these are all from CF sputum      KEEP
# PRJEB14149; one CF sample                     KEEP
# PRJNA437845; some of these are burn samples   REMOVE
# PRJEB38221; this includs healthy, will filter KEEP
# PRJNA931830; these are all from CF OP swabs   KEEP
# PRJEB51171; these are all from CF airways     KEEP
# PRJNA510441; these are all from CF sputum     KEEP
# PRJNA644285; these are CF sputum              KEEP
# PRJNA825831; all CF respiratory samples       KEEP
# PRJEB41644; throat swab and CF sputum         KEEP
# PRJNA846291; all CF airway                    KEEP

In [ ]:
# remove studies with ambiguous organisms
sra_ambig_org_filt = (
    sra_org_filt.filter(~pl.col('bioproject').is_in(['PRJNA437845', 'PRJEB28158']))
)
print("Number of ambiguous organism filtered sequences", len(sra_ambig_org_filt))

In [ ]:
# remove artificial media samples
# download SRA run list from NCBI
PRJNA861321_df = pl.read_csv('bioproject_metadata/PRJNA861321.csv')
PRJNA861321_remove = set(PRJNA861321_df.filter(pl.col('experimental_condition') != 'Source')['Run'])
sra_media_filt = (
    sra_ambig_org_filt
        .filter(~pl.col('acc').is_in(PRJNA861321_remove))
)
print("Number of media filtered sequences", len(sra_media_filt))

In [ ]:
# identify non-cf samples
PRJEB38221_df = pl.read_csv('bioproject_metadata/PRJEB38221.csv')
noncf_samples = set(PRJEB38221_df.filter(pl.col('Sample_name').str.contains('_H'))['Run'])

PRJNA1055940_df = pl.read_csv('bioproject_metadata/PRJNA1055940.csv')
noncf_samples = noncf_samples.union(set(PRJNA1055940_df.filter(pl.col('Host_disease') != 'Cystic fibrosis')['Run']))

PRJNA316588_df = pl.read_csv('bioproject_metadata/PRJNA316588.csv')
noncf_samples = noncf_samples.union(set(PRJNA316588_df.filter(~pl.col('Sample Name').str.contains('CF-'))['Run']))

# remove non-cf samples from the dataframe
sra_noncf_filt = sra_media_filt.filter(~pl.col('acc').is_in(noncf_samples))
print("Number of non-CF filtered sequences", len(sra_noncf_filt))

In [ ]:
# remove bioprojects with less than 10 samples after filters
large_bioprojects = (
    sra_noncf_filt
        .group_by('bioproject')
        .agg(pl.count('acc').alias('count'))
        .filter(pl.col('count') >= 10)
)

In [ ]:
# calculate number of samples of each type from different studies
sample_type_dict = {
    'PRJNA1101448': 'oropharyngeal swab',
    'PRJNA931830': 'oropharyngeal swab',
    'PRJNA530252': 'sputum',
    'PRJEB54014': 'sputum',
    'PRJNA516870': 'sputum',
    'PRJNA516442': 'sputum',
    'PRJEB51171': 'deep cough swabs',
    'PRJEB56242': 'sputum',
    'PRJEB50802': 'unknown',
    'PRJEB32062': 'sputum',
    'PRJNA1091195': 'sputum',
    'PRJNA510441': 'unknown',
    'PRJNA839435': 'sputum',
    'PRJNA644285': 'sputum',
    'PRJNA316056': 'sputum',
    'PRJEB38221': 'deep cough swabs',
    'PRJNA316588': 'sputum',
    'PRJNA861321': 'sputum'
}

# calculate number of individuals from each study
individual_count_dict = {
    'PRJNA1101448': 20,
    'PRJNA931830': 15,
    'PRJEB52317': 65,
    'PRJNA530252': 30,
    'PRJNA825831': 22,
    'PRJNA861321': 11,
    'PRJNA846291': 20,
    'PRJEB54014': 40,
    'PRJNA516870': 22,
    'PRJNA516442': 8,
    'PRJEB51171': 31,
    'PRJEB56242': 26,
    'PRJEB50802': 12,
    'PRJEB32062': 4,
    'PRJNA1091195': 6,
    'PRJNA510441': 1,
    'PRJNA839435': 6,
    'PRJNA644285': 3,
    'PRJNA316056': 12,
    'PRJEB38221': 42,
    'PRJNA1055940': 13,
    'PRJNA316588': 6
}

# load metadata to determine number of samples of each type
PRJEB52317_df = pl.read_csv('bioproject_metadata/PRJEB52317.csv')
sample_type_dict.update(PRJEB52317_df.set_index('Run')['environment_(material)'].to_dict())

PRJNA825831_df = pl.read_csv('bioproject_metadata/PRJNA825831.csv')
sample_type_dict.update(PRJNA825831_df.set_index('Run')['isolation_source'].to_dict())

PRJNA846291_df = pl.read_csv('bioproject_metadata/PRJNA846291.csv')
PRJNA846291_df['isolation_source'] = PRJNA846291_df['isolation_source'] .replace('swab', 'oropharyngeal swab')
sample_type_dict.update(PRJNA846291_df.set_index('Run')['isolation_source'].to_dict())

PRJNA1055940_df = pl.read_csv('bioproject_metadata/PRJNA1055940.csv')
sample_type_dict.update(PRJNA1055940_df.set_index('Run')['env_local_scale'].to_dict())
